In [ ]:
# Let us import some useful libraries to start with!
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
# Let us import the famous titanic dataset from Kagle (https://www.kaggle.com/c/titanic). We will try to predict
# by classification: survival or deceased, by implementing Logistic Regression model
# in Python for classification.
# We will use an "almost-cleaned" version of the titanic data set.
# Note: The original version of this dataset hosted directly on Kaggle will need some additional cleaning.

train = pd.read_csv('titanic_train.csv')

In [ ]:
train.head(10)

In [ ]:
train.info()

Let us do some basic EDA

In [ ]:
# Looking into missing values/data
#sns.heatmap(train.isnull(),cmap='viridis')
sns.heatmap(train.isnull(),yticklabels=False,cbar=False,cmap='viridis')

In [ ]:
# it helps to find out the ratio of the target labels:
sns.set_style('whitegrid')

#sns.countplot(x='Survived',data=train,palette='RdBu_r')
sns.countplot(x='Survived', data=train, hue='Sex',palette='RdBu_r')

In [ ]:
sns.countplot(x='Survived', data=train, hue='Pclass',palette='rainbow')

In [ ]:
#sns.distplot(train['Age'].dropna(),kde=False,color='darkred',bins=30)
sns.displot(train['Age'].dropna(),kde=True,color='darkred',bins=30)

In [ ]:
train['Age'].hist(bins=30,color='darkred',alpha=0.7)

In [ ]:
sns.countplot(x='SibSp',data=train)

In [ ]:
train['Fare'].hist(color='green',bins=40,figsize=(8,4))

In [ ]:
import cufflinks as cf
cf.go_offline()

In [ ]:
train['Fare'].iplot(kind='hist',bins=30,color='green')

#Data Cleaning:
We want to fill in missing age data instead of just dropping the missing age data rows. 
One way to do this is by filling in the mean age of all the passengers (imputation). 
However we can be smarter about this and check the average age by passenger class. For example:

In [ ]:
plt.figure(figsize=(12, 7))
sns.boxplot(x='Pclass',y='Age',data=train,palette='winter')

In [ ]:
# Confirm the average values to be used in an imputation function
train.groupby(by='Pclass', axis=0)['Age'].mean()

Let us write a simple fxn to impute the missing ages:

In [ ]:
def impute_age(cols):
    Age = cols[0]
    Pclass = cols[1]
    
    if pd.isnull(Age):

        if Pclass == 1:
            return 38

        elif Pclass == 2:
            return 30

        else:
            return 25

    else:
        return Age

In [ ]:
train['Age'] = train[['Age','Pclass']].apply(impute_age,axis=1)

In [ ]:
sns.heatmap(train.isnull(),yticklabels=False,cbar=False,cmap='viridis')

In [ ]:
train.drop('Cabin',axis=1,inplace=True)

In [ ]:
train.head()

In [ ]:
train.dropna(inplace=True)

## Converting Categorical Features¶
We'll need to convert categorical features to dummy variables using pandas! Otherwise our machine learning algorithm won't be able to directly take in those features as inputs.

In [ ]:
train.info()

In [ ]:
sex = pd.get_dummies(train['Sex'],drop_first=True)
embark = pd.get_dummies(train['Embarked'],drop_first=True)
#pclass = pd.get_dummies(train['Pclass'],drop_first=True)

In [ ]:
#embark.head()
sex.head()

In [ ]:
# drop col we have created dummies for; also fields with no apparent information for modeling purposes
train.drop(['Sex','Embarked','Name','Ticket'],axis=1,inplace=True)

In [ ]:
# attach the created dummy variables to the main data frame
train = pd.concat([train, sex, embark], axis=1)

In [ ]:
train.head()

Building a Logistic Regression model:
Let's start by splitting our data into a training set and test set

Train Test Split:

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(train.drop('Survived',axis=1), 
                                                    train['Survived'], test_size=0.30, 
                                                    random_state=101)

## Training and Predicting

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
# Instantiate and fit a Logistic Regression model to the data
logmodel = LogisticRegression(max_iter=1000)
logmodel.fit(x_train,y_train)

In [ ]:
predictions = logmodel.predict(x_test)

## Model Evaluation
We can check precision, recall, f1-score using classification report!

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
print(classification_report(y_test,predictions))

# Getting The Associated Confusion Matrix:


In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
print(confusion_matrix(y_test,predictions))